# Resultados — `MIL-CREDA` (v1)

Informe ejecutado, no la fuente de la verdad: cada afirmación matemática vive como una prueba bajo `tests/`, y este cuaderno corre las mediciones que esas pruebas ya validan y muestra la evidencia. Reemplaza a los cuadernos de informe y de análisis latente que existían por separado -- `Benchmark_Report_v1.ipynb` y `Benchmark_Latent_v1.ipynb` -- en uno solo, porque las seis secciones de abajo leen el mismo registro y se citan entre sí.

**Todavía no corrió ninguna campaña.** Cada sección lo dice donde le falten sus propios datos, en vez de fallar: lo que sigue es la forma del informe, no un resultado.

Seis secciones, en este orden: el barrido de ruido sobre el piso solo; exactitud de fuente por brazo; exactitud de destino por brazo; los mecanismos de atención sobre el método completo; la representación (geometría, separabilidad, rejillas latentes, correspondencia entre bolsas); y las curvas de pérdida, sólo para comprobar la normalización.

In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# El repositorio ya lo resolvió la celda de arriba -- la de la forja, byte por byte -- y esta sólo lo usa.
import json
import sys
from pathlib import Path

import torch
from IPython.display import Markdown, display

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))

from MIL_CREDA_Benchmark import config, figures, harness, latent, tables


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo."""
    display(Markdown(text))


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIGURES = config.PRODUCT / "Results" / "figures"
print("repository:", REPOSITORY)
print("device:", DEVICE)

In [ ]:
def cargar_corridas(rate: float = 0.0, kind: str = "campaign", pilot: bool = False):
    """Las corridas y la reducción de un árbol de resultados, o `(None, None,
    ruta, ensayo)` si esa corrida todavía no dejó registro en ninguna escala.

    La corrida completa le gana siempre al ensayo, y sólo si la completa no
    existe se mira el ensayo -- la misma regla que
    `harness.search_source_note` ya aplica a la búsqueda de techos, acá
    para la campaña. `ensayo` viaja junto al resultado, nunca implícito, para
    que cada celda que dibuja pesos (`latent.latent_grid`, `checkpoint_for`,
    `available`) pueda pasarle la MISMA escala con la que se leyó esto: un
    ensayo que mira el árbol completo, o al revés, dibuja paneles apagados
    sin un solo error que lo delate.

    Glue de este cuaderno y no del banco: pertenece acá porque decide si una
    sección puede correr, y ninguna otra cosa del repositorio necesita
    preguntarlo.
    """
    for ensayo in (False, True):
        raiz = config.results_for(rate, kind, ensayo)
        runs_path = raiz / "runs.jsonl"
        resumen_path = raiz / "summary.json"
        if runs_path.exists() and resumen_path.exists():
            corridas = [json.loads(linea) for linea in
                       runs_path.read_text(encoding="utf-8").splitlines() if linea.strip()]
            resumen = json.loads(resumen_path.read_text(encoding="utf-8"))
            return corridas, resumen.get("reduction", {}), runs_path, ensayo
    return None, None, runs_path, False

## El sello

Los límites de la corrida, dichos una vez y calculados, antes de cualquier número de abajo. La corrida completa le gana siempre al ensayo; si lo que sigue lee del ensayo, lo dice el propio sello de `tables.stamp`.

In [ ]:
corridas, reduccion, runs_path, ES_ENSAYO = cargar_corridas()
# De qué árbol salieron los números de abajo -- completo o ensayo --
# dicho siempre y no sólo cuando algo anda mal, el mismo mecanismo que
# `harness.search_source_note` ya usa para los techos.
show(harness.campaign_source_note(ES_ENSAYO if reduccion is not None else None))
if reduccion is not None:
    show(tables.stamp(reduccion, markdown=True))
else:
    show(
        f"**Sin corrida todavía.** No existe `{runs_path.relative_to(ROOT)}`: "
        f"la campaña no corrió, ni a escala completa ni de ensayo, así que no "
        f"hay límites que declarar. Cada sección de abajo lo dice de nuevo "
        f"donde le falten sus propios datos.")
    # El protocolo declarado, no una medición: lo único que las secciones de
    # abajo pueden citar antes de que exista una corrida real.
    reduccion = {"seeds": config.SEEDS, "epochs": config.EPOCHS,
                "backbone": config.BACKBONE, "revision": config.REVISION}

## 1 · El barrido de ruido — el piso solo

Abre el informe porque responde la pregunta más simple: si el ruido en las etiquetas de entrenamiento perjudica incluso al método que **no adapta**. `MIL-Baseline` (o el piso que `config.FLOOR_OF` declare, nunca un id fijado acá) entrena sólo con la pérdida supervisada de fuente, así que una caída en esta curva no puede venir de ningún término de adaptación -- es lo que el ruido le hace a la propia representación. Dos figuras, fuente y destino, con la misma curva.

In [ ]:
# Los pisos declarados hoy, nunca un id escrito acá.
pisos_declarados = sorted(set(config.FLOOR_OF.values()))

### 1a · Fuente

El piso, medido sobre el dominio en el que entrena. Como no adapta, no hay término que pudiera compensar una caída acá: lo que se ve es exactamente lo que la contaminación de las etiquetas de entrenamiento le hace a la representación, sin nada más mezclado en la lectura. Sirve de referencia para la figura de destino que sigue: si la fuente cae poco y el destino cae mucho, la brecha entre dominios es lo que se está ensanchando y no la calidad general del modelo.

In [ ]:
show(tables.objective("noise.floor.source"))

In [ ]:
figura_fuente = figures.noise_curves(
    "sourceAccuracy", path=FIGURES / "noise_floor_source.pdf", arms=tuple(pisos_declarados))
display(figures.inline(figura_fuente))

In [ ]:
show(tables.conclusion_noise_floor("sourceAccuracy"))

### 1b · Destino

El piso, medido sobre el dominio que nunca ve durante el entrenamiento. Es la lectura que más importa de esta sección: un piso entrenado sólo con fuente no tiene ningún mecanismo de adaptación al que la contaminación pudiera llegarle indirectamente, así que cualquier caída acá viene enteramente de que la representación aprendida en fuente generaliza peor a destino cuando el material de entrenamiento está sucio.

In [ ]:
show(tables.objective("noise.floor.target"))

In [ ]:
figura_destino = figures.noise_curves(
    "targetAccuracy", path=FIGURES / "noise_floor_target.pdf", arms=tuple(pisos_declarados))
display(figures.inline(figura_destino))

In [ ]:
show(tables.conclusion_noise_floor("targetAccuracy"))

## 2 · Exactitud de fuente, por brazo

Complemento de la sección siguiente: un método que gana en destino destruyendo la fuente es el caso degenerado, y esta tabla es lo único que lo distingue de un éxito real. Dos bloques -- `sin` ruido y `con` -- con la columna `Ruido` adelante, y un puesto por bloque: dos brazos comparten puesto cuando la diferencia entre sus promedios no supera su error estándar combinado, es decir, cuando esta corrida no alcanza a distinguirlos. Los dos bloques rankean por separado: compartir puesto en `sin` no dice nada sobre `con`.

In [ ]:
show(tables.objective("sourceAccuracy"))

In [ ]:
if corridas is not None:
    show(tables.render(corridas, "sourceAccuracy", reduccion,
                       rate=config.NOISE_REPORTED, markdown=True))
    show(tables.conclusion_with_noise(corridas, "sourceAccuracy", reduccion,
                                    config.NOISE_REPORTED))
else:
    show(f"**Sin corrida todavía.** No existe `{runs_path.relative_to(ROOT)}`: "
         f"no hay exactitud de fuente que mostrar.")

## 3 · Exactitud de destino, por brazo

La cabecera del informe: misma forma que la sección anterior, dos bloques y un puesto por bloque, leída **junto a** la tabla de fuente y nunca sola -- una subida acá pagada con una caída allá no es adaptación.

In [ ]:
show(tables.objective("targetAccuracy"))

In [ ]:
if corridas is not None:
    show(tables.render(corridas, "targetAccuracy", reduccion,
                       rate=config.NOISE_REPORTED, markdown=True))
    show(tables.conclusion_with_noise(corridas, "targetAccuracy", reduccion,
                                    config.NOISE_REPORTED))
else:
    show(f"**Sin corrida todavía.** No existe `{runs_path.relative_to(ROOT)}`: "
         f"no hay exactitud de destino que mostrar.")

In [ ]:
# `render_mechanisms`/`conclusion_mechanisms` ya dicen llanamente que el
# barrido no corrió cuando el registro falta o no declara mecanismos, así
# que se llaman siempre y no adentro de un `if`: eso es lo que las hace
# ejercitadas de verdad y no una rama que nadie recorre todavía. Cargado
# antes del encabezado de la sección para que nada se interponga entre el
# encabezado y la primera lectura que enmarca.
# Por la misma puerta de la que salieron las corridas de arriba, y con la
# misma respuesta: el barrido de mecanismos escribe en el árbol de ensayo
# cuando corre en ensayo, así que una ruta fija a la corrida completa
# dibujaría la Sección 4 como ausente al lado de tablas de piloto que sí
# tienen números. Leer de un árbol lo de una sección y de otro lo de otra
# es cómo un cuaderno termina mezclando dos experimentos en una página.
mechanism_record_path = (config.results_for(0.0, "campaign", ES_ENSAYO)
                         / Path(tables.MECHANISM_RECORD).name)
mechanism_record = (json.loads(mechanism_record_path.read_text(encoding="utf-8"))


## 4 · Mecanismos de atención — sobre el método completo

Compara mecanismos y no brazos: siempre el método completo (`MIL-CREDA`), con la atención de la Sección 3 tal como este repositorio la implementa contra ABMIL tal como fue publicado (`v_R` sin normalizar), ABMIL con compuerta, `max` y `mean`. Ningún nombre de mecanismo vive en `tables.py`: vienen enteros del registro (`tables.MECHANISM_RECORD`), y si ese registro no existe la sección lo dice en vez de inventar una fila. Dos tablas, fuente y destino.

### 4a · Fuente

Cada mecanismo, medido sobre el dominio en el que el brazo completo entrena. El complemento de la lectura de destino que sigue: un mecanismo que gana en destino hundiendo la fuente no está aportando nada que valga la pena, y esta tabla es la única forma de verlo.

In [ ]:
show(tables.objective("sourceAccuracy"))

In [ ]:
show(tables.render_mechanisms(mechanism_record, "sourceAccuracy", markdown=True))
show(tables.conclusion_mechanisms(mechanism_record, "sourceAccuracy"))

### 4b · Destino

Cada mecanismo, medido sobre el dominio que la adaptación existe para mejorar. La lectura que más importa de esta sección: si el mecanismo de atención de este repositorio queda adelante o atrás de ABMIL publicado, de su variante con compuerta, o de un `max`/`mean` sin atención en absoluto.

In [ ]:
show(tables.objective("targetAccuracy"))

In [ ]:
show(tables.render_mechanisms(mechanism_record, "targetAccuracy", markdown=True))
show(tables.conclusion_mechanisms(mechanism_record, "targetAccuracy"))

## 5 · Representación

Si el método alinea las clases entre dominios, o sólo mueve la escala del espacio. En orden: dos tablas cuantitativas (agrupamiento y separabilidad de dominio), cada una limpia y contaminada; la rejilla latente, limpia y contaminada; la figura de bolsas con sus aciertos por brazo; y por último la correspondencia bolsa por bolsa -- de destino a fuente, y su inversa.

In [ ]:
# Checkpoints declarados, de esta revisión, con hiperparámetros vigentes y
# que son la mediana propia de su celda -- nunca los extras que una
# promoción apareada agregó para el piso de otro brazo. `ES_ENSAYO` viaja de
# la celda del sello: leer los pesos de una escala mientras se leyó la
# corrida de otra dibujaría paneles apagados sin un solo error que lo avise.
def checkpoints_medianos(rate: float = 0.0, pilot: bool = ES_ENSAYO):
    encontrados = latent.available(rate, pilot)
    return [c for c in encontrados if c["median"] and c["declared"]
           and c["currentRevision"] and c["currentHyperparameters"]]


checkpoints_limpios = checkpoints_medianos(0.0)
checkpoints_sucios = checkpoints_medianos(config.NOISE_REPORTED)
if not checkpoints_limpios:
    show(
        f"**Sin checkpoints todavía.** "
        f"`{config.models_for(0.0, 'campaign', ES_ENSAYO).relative_to(ROOT)}` no "
        f"tiene manifiestos: la campaña no dejó pesos. No es una lectura "
        f"vacía: es que la corrida no existe. El resto de esta sección no "
        f"puede correr sin al menos un checkpoint.")
else:
    # Medido acá, antes del encabezado de 5a, para que nada se interponga
    # entre ese encabezado y la primera lectura que enmarca.
    lecturas_limpias = [latent.analyse(c, DEVICE) for c in checkpoints_limpios]
    lecturas_sucias = [latent.analyse(c, DEVICE) for c in checkpoints_sucios]

### 5a-i · Agrupamiento (razón de distancias)

La razón `cruzada / entre clases` de la Sección 5 del paper, leída en el RKHS que el método alinea y nunca en el embedding euclídeo: baja tanto si la misma clase se juntó entre dominios como si el espacio entero se encogió, y separar esas dos lecturas es lo que la conclusión hace.

In [ ]:
show(tables.objective("geometry.ratio"))

In [ ]:
if checkpoints_limpios:
    show(tables.render_readings(lecturas_limpias, "geometry.ratio",
                                "razón de distancias", contaminated=lecturas_sucias,
                                rate=config.NOISE_REPORTED, markdown=True))
    show(tables.conclusion_readings_with_noise(
        lecturas_limpias, lecturas_sucias, "geometry.ratio", config.NOISE_REPORTED)
        if lecturas_sucias else tables.conclusion_geometry(lecturas_limpias))

### 5a-ii · Separabilidad de dominio

Qué tan bien un clasificador lineal distingue fuente de destino en el espacio alineado: cerca del azar es lo buscado, porque significa que la información de dominio dejó de estar disponible. Complementa la razón de distancias de arriba -- una mide si las clases se alinearon, ésta mide si los dominios se volvieron indistinguibles.

In [ ]:
show(tables.objective("domainSeparability"))

In [ ]:
if checkpoints_limpios:
    show(tables.render_readings(lecturas_limpias, "domainSeparability",
                                "separabilidad de dominio", contaminated=lecturas_sucias,
                                rate=config.NOISE_REPORTED, markdown=True))
    show(tables.conclusion_readings_with_noise(
        lecturas_limpias, lecturas_sucias, "domainSeparability", config.NOISE_REPORTED)
        if lecturas_sucias else tables.conclusion_separability(lecturas_limpias))

In [ ]:
if checkpoints_limpios:
    corridas_para_semilla = corridas if corridas is not None else [
        {"seed": c["seed"], "targetAccuracy": c["targetAccuracy"]} for c in checkpoints_limpios]
    semilla_exhibicion = latent.display_seed(corridas_para_semilla)
    transferencias_grilla = tables.best_transfers(corridas) if corridas else [
        f"{s}->{t}" for s, t in config.VERDICT_TRANSFERS[:config.FIGURE_TRANSFER_COUNT]]

### 5b · Rejilla latente, limpia

El espacio compartido de la Sección 5a, mostrado y no sólo medido: una fila por transferencia -- las que la campaña alcanzó más alto, la misma regla que las figuras de curvas -- y una columna por método de `config.LATENT_PANELS`, todos dibujados a nivel de instancia porque es el único espacio que cada arm tiene. La semilla es la de exhibición, elegida por una regla que no favorece a ningún método.

In [ ]:
show(tables.objective("latent.grid.clean"))

In [ ]:
if checkpoints_limpios:
    rejilla = latent.latent_grid(
        FIGURES / "latent_grid_clean.pdf", config.LATENT_PANELS,
        transferencias_grilla, semilla_exhibicion, DEVICE, rate=0.0,
        pilot=ES_ENSAYO)
    display(figures.inline(rejilla))

In [ ]:
if checkpoints_limpios:
    show(tables.conclusion_distances(lecturas_limpias))

### 5b (cont.) · Rejilla latente, contaminada

La misma rejilla, sobre los checkpoints entrenados con las etiquetas de entrenamiento contaminadas. Comparada panel por panel contra la de arriba y no por separado, porque lo que importa es cuánta mezcla entre clases sobrevive y no cómo se ve cada una por sí sola.

In [ ]:
show(tables.objective("latent.grid.noisy"))

In [ ]:
if checkpoints_limpios and checkpoints_sucios:
    rejilla_sucia = latent.latent_grid(
        FIGURES / "latent_grid_noisy.pdf", config.LATENT_PANELS,
        transferencias_grilla, semilla_exhibicion, DEVICE,
        rate=config.NOISE_REPORTED, pilot=ES_ENSAYO)
    display(figures.inline(rejilla_sucia))
elif checkpoints_limpios:
    show(f"**Sin checkpoints contaminados a ρ={config.NOISE_REPORTED:g}.** "
         f"La rejilla limpia queda sola.")

In [ ]:
if checkpoints_limpios and checkpoints_sucios:
    show(tables.conclusion_distances(lecturas_sucias))

### 5c · La figura de bolsas, limpia

Tres columnas -- el piso, el mismo método sin término local y el completo, elegidas por el mecanismo (`config.BAG_PANELS`) y no por la clasificación -- con la misma bolsa mediana de cada clase destacada en cada panel. Los números de la figura, como tabla y no como pie de imagen, donde se comparan de una fila a otra.

In [ ]:
show(tables.objective("correspondence.grid.clean"))

In [ ]:
if checkpoints_limpios:
    rejilla_bolsas = latent.correspondence_grid(
        FIGURES / "correspondence_grid_clean.pdf", config.BAG_PANELS,
        transferencias_grilla, semilla_exhibicion, DEVICE, rate=0.0,
        pilot=ES_ENSAYO)
    display(figures.inline(rejilla_bolsas["figure"]))

### 5c-i · Los aciertos, en números

Vivían dentro de la figura de arriba, como pie de cada panel. Ahí obligaban a leer una cifra dentro de un dibujo, que es donde peor se compara, y de paso hacían que las tres columnas se vieran igual de afirmativas. La figura queda para lo que hace bien -- mostrar si los sujetos se asocian -- y los números quedan acá, donde se comparan de una fila a otra.

In [ ]:
show(tables.objective("correspondence"))

In [ ]:
if checkpoints_limpios:
    show(tables.render_correspondence(rejilla_bolsas["scored"], markdown=True))
    show(tables.conclusion_correspondence(rejilla_bolsas["scored"]))

### 5c (cont.) · La figura de bolsas, contaminada

La misma figura sobre la campaña contaminada, con la misma tabla de aciertos al lado. Leída junto a la limpia: si la correspondencia local sobrevive al ruido mejor que la global es exactamente lo que esta comparación existe para mostrar, y es una lectura distinta de las dos tablas de la Sección 5a -- ésas miden geometría global, ésta mide si el emparejamiento bolsa a bolsa sigue siendo correcto.

In [ ]:
show(tables.objective("correspondence.grid.noisy"))

In [ ]:
if checkpoints_limpios and checkpoints_sucios:
    rejilla_bolsas_sucia = latent.correspondence_grid(
        FIGURES / "correspondence_grid_noisy.pdf", config.BAG_PANELS,
        transferencias_grilla, semilla_exhibicion, DEVICE,
        rate=config.NOISE_REPORTED, pilot=ES_ENSAYO)
    display(figures.inline(rejilla_bolsas_sucia["figure"]))
elif checkpoints_limpios:
    show(f"**Sin checkpoints contaminados a ρ={config.NOISE_REPORTED:g}.** "
         f"La figura de bolsas limpia queda sola.")

### 5c-ii · Los aciertos, en números, contaminados

La misma tabla de aciertos sobre la campaña contaminada, leída junto a la limpia de arriba y nunca por separado: si el aporte del término local sobrevive al ruido es la comparación fila a fila entre las dos.

In [ ]:
show(tables.objective("correspondence"))

In [ ]:
if checkpoints_limpios and checkpoints_sucios:
    show(tables.render_correspondence_contaminated(
        rejilla_bolsas_sucia["scored"], config.NOISE_REPORTED, markdown=True))
    show(tables.conclusion_correspondence(rejilla_bolsas_sucia["scored"]))

### 5d · Bolsas de destino y sus 5 bolsas fuente más cercanas

El brazo completo, sobre la transferencia y semilla de exhibición: por cada bolsa de evaluación de destino, sus cinco bolsas fuente de entrenamiento más cercanas por el kernel de bolsa de la Sección 3 -- sobre TODAS las bolsas fuente, nunca un subconjunto -- marcando cuáles son de la clase verdadera.

In [ ]:
show(tables.objective("correspondence"))

In [ ]:
if checkpoints_limpios:
    arm_completo = next(
        (a for a in config.BAG_PANELS if config.ARMS_BY_ID[a]["local"]), None)
if checkpoints_limpios and arm_completo is None:
    show("Ningún brazo de `config.BAG_PANELS` declara término local: no hay "
         "método completo sobre el que medir la correspondencia.")
elif checkpoints_limpios:
    checkpoint_completo = latent.checkpoint_for(
        arm_completo, transferencias_grilla[0], semilla_exhibicion, rate=0.0,
        pilot=ES_ENSAYO)
    if checkpoint_completo is None:
        show(f"**Sin checkpoint de `{config.NAME_OF[arm_completo]}`** en "
             f"`{transferencias_grilla[0]}` semilla {semilla_exhibicion}: no hay "
             f"correspondencia que medir para esa celda todavía.")
    else:
        modelo, fuente, destino = latent.load(checkpoint_completo, DEVICE)
        referencia = latent.bag_pairs(modelo, fuente, destino, DEVICE)
        vecinos = latent.top_k_source_bags(referencia, k=5)
        show(tables.render_bag_neighbors(vecinos, markdown=True))
        show(tables.conclusion_bag_neighbors(vecinos))

### 5e · La tabla inversa: cuántas bolsas de destino usaron cada bolsa fuente

Expone las bolsas fuente que el top-5 de nadie nombra -- algo que la tabla anterior, indexada al revés, no puede mostrar.

In [ ]:
show(tables.objective("correspondence"))

In [ ]:
if checkpoints_limpios and arm_completo is not None and checkpoint_completo is not None:
    uso = latent.source_bag_usage(referencia, k=5)
    show(tables.render_source_bag_usage(uso, markdown=True))
    show(tables.conclusion_source_bag_usage(uso))

## 6 · Curvas de pérdida — sólo para comprobar la normalización

No para leer una trayectoria: para comprobar que los términos de la Ec. (39) se quedan en la misma escala [0, 1) que la normalización promete. La banda sombreada de cada figura es exactamente ese intervalo, y la conclusión que sigue a cada una no dice qué brazo tiene la curva más baja -- dice si algún punto, de algún brazo, de alguna corrida, se salió del intervalo. Dos figuras, el término de adaptación y el supervisado.

### 6a · Término de adaptación

Cada brazo adaptado (`config.FLOOR_OF`, nunca un id escrito acá), un panel por transferencia. Section 5's arm completo aparte no importa acá: lo que se comprueba es si la construcción de la Sección 5 del paper mantiene el término dentro de [0, 1) para TODOS los brazos que lo llevan, no sólo para el que gana.

In [ ]:
show(tables.objective("adaptation"))

In [ ]:
if corridas is not None:
    figura_adaptacion = figures.adaptation_curves(
        FIGURES / "adaptation_curves.pdf",
        arms=tuple(sorted(config.FLOOR_OF)), transfers=tables.best_transfers(corridas))
    display(figures.inline(figura_adaptacion))
else:
    show(f"**Sin corrida todavía.** No existe `{runs_path.relative_to(ROOT)}`: "
         f"no hay curva de adaptación que dibujar.")

In [ ]:
if corridas is not None:
    show(tables.conclusion_normalization(corridas, "adaptation"))

### 6b · Término supervisado

Cada piso y cada brazo adaptado (la unión de `config.FLOOR_OF` y sus valores), un panel por transferencia. Es donde se vería un término de adaptación que desestabiliza el ajuste: si la curva supervisada se deshace justo en los brazos que llevan adaptación y no en los pisos, el término de adaptación lo rompió.

In [ ]:
show(tables.objective("supervised"))

In [ ]:
if corridas is not None:
    arms_supervisado = tuple(sorted({*config.FLOOR_OF, *config.FLOOR_OF.values()}))
    figura_supervisada = figures.supervised_curves(
        FIGURES / "supervised_curves.pdf",
        arms=arms_supervisado, transfers=tables.best_transfers(corridas))
    display(figures.inline(figura_supervisada))
else:
    show(f"**Sin corrida todavía.** No existe `{runs_path.relative_to(ROOT)}`: "
         f"no hay curva supervisada que dibujar.")

In [ ]:
if corridas is not None:
    show(tables.conclusion_normalization(corridas, "supervised"))

## El sello del código

In [ ]:
# Contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())